# 04b - ConvLSTM U-Net variable temporal stride experiments

This notebook reruns the ConvLSTM U-Net temporal baseline with wider five-frame windows while preserving the original preprocessing, official EchoNet TRAIN/VAL/TEST split, model architecture, and center-frame segmentation target. Test evaluation is performed only after each stride's best checkpoint has been selected using validation Dice.

In [ ]:
# Kaggle setup. Skip this cell when the environment already satisfies requirements.txt.
%pip install -q monai opencv-python-headless pandas matplotlib tqdm

In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

# On Kaggle, set PROJECT_ROOT to the directory containing src/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import split_by_echonet_filelist
from src.temporal_dataset_variable_stride import (
    EchoNetTemporalVariableStrideDataset,
    build_fps_lookup,
    load_temporal_metadata,
)
from src.temporal_model import build_convlstm_unet
from src.temporal_train_version_2 import (
    evaluate_temporal_test_v2,
    fit_temporal_v2,
    get_temporal_loss,
    plot_temporal_history,
    save_json,
    write_experiment_log,
)
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get('ECHONET_RAW_DIR', PROJECT_ROOT / 'data' / 'raw' / 'EchoNet-Dynamic'))
PROCESSED_DIR = Path(os.environ.get('ECHONET_PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed'))
VIDEOS_DIR = RAW_DIR / 'Videos'
BASE_RUN_DIR = Path('/kaggle/working/outputs/runs/convlstm_unet_variable_strides') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet_variable_strides'
ORIGINAL_BASELINE_RUN_DIR = Path(os.environ.get('CONVLSTM_STRIDE1_RUN_DIR', PROJECT_ROOT / 'outputs' / 'runs' / 'convlstm_unet'))
if not ORIGINAL_BASELINE_RUN_DIR.exists():
    fallback_baseline = PROJECT_ROOT / 'outputs' / 'runs' / 'ConvLSTM_Unet_06_11'
    if fallback_baseline.exists():
        ORIGINAL_BASELINE_RUN_DIR = fallback_baseline
BASE_RUN_DIR.mkdir(parents=True, exist_ok=True)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Raw EchoNet directory: {RAW_DIR}')
print(f'Processed dataset directory: {PROCESSED_DIR}')
print(f'Variable-stride output directory: {BASE_RUN_DIR}')
print(f'Optional stride-1 baseline directory: {ORIGINAL_BASELINE_RUN_DIR}')

## Configuration

In [ ]:
RUN_MODE = 'smoke'  # change to 'full' for complete stride experiments
TEMPORAL_STRIDES = [4, 6, 8, 10]

SMOKE_CONFIG = {
    'run_mode': 'smoke',
    'seed': 42,
    'sequence_length': 5,
    'image_size': [112, 112],
    'epochs': 1,
    'early_stopping_patience': 3,
    'batch_size': 4,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'max_train_samples': 32,
    'max_val_samples': 16,
    'max_test_samples': 16,
    'channels': [16, 32, 64, 128],
    'max_prediction_examples': 10,
    'max_area_curve_videos': 12,
}

FULL_CONFIG = {
    'run_mode': 'full',
    'seed': 42,
    'sequence_length': 5,
    'image_size': [112, 112],
    'epochs': 50,
    'early_stopping_patience': 10,
    'batch_size': 8,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'max_train_samples': None,
    'max_val_samples': None,
    'max_test_samples': None,
    'channels': [16, 32, 64, 128],
    'max_prediction_examples': 10,
    'max_area_curve_videos': 12,
}

base_config = SMOKE_CONFIG if RUN_MODE == 'smoke' else FULL_CONFIG
print(base_config)

## Load processed masks and official EchoNet splits

In [ ]:
metadata_path = PROCESSED_DIR / 'metadata.csv'
assert metadata_path.exists(), 'Run notebook 02 or mount the reusable processed Kaggle Dataset.'
assert VIDEOS_DIR.exists(), f'Videos directory not found: {VIDEOS_DIR}'

samples = load_temporal_metadata(metadata_path)
file_list, _ = load_echonet_tables(RAW_DIR)
fps_by_video = build_fps_lookup(file_list)
train_samples, val_samples, test_samples = split_by_echonet_filelist(samples, file_list)

matched_count = len(train_samples) + len(val_samples) + len(test_samples)
assert matched_count == len(samples), (
    f'{len(samples) - matched_count} samples did not match the official EchoNet split.'
)
assert min(len(train_samples), len(val_samples), len(test_samples)) > 0, 'Every official split must contain samples.'

if base_config['max_train_samples'] is not None:
    train_samples = train_samples[:base_config['max_train_samples']]
if base_config['max_val_samples'] is not None:
    val_samples = val_samples[:base_config['max_val_samples']]
if base_config['max_test_samples'] is not None:
    test_samples = test_samples[:base_config['max_test_samples']]

print(f'Total processed labeled frames: {len(samples):,}')
print(f'Train samples: {len(train_samples):,}')
print(f'Validation samples: {len(val_samples):,}')
print(f'Test samples: {len(test_samples):,}')
print(f'Videos with FPS metadata: {len(fps_by_video):,}')

## Train validation-selected stride experiments

In [ ]:
def make_loaders(temporal_stride: int):
    dataset_kwargs = {
        'videos_dir': VIDEOS_DIR,
        'sequence_length': base_config['sequence_length'],
        'temporal_stride': temporal_stride,
        'image_size': tuple(base_config['image_size']),
        'fps_by_video': fps_by_video,
    }
    train_dataset = EchoNetTemporalVariableStrideDataset(train_samples, augment=True, **dataset_kwargs)
    val_dataset = EchoNetTemporalVariableStrideDataset(val_samples, augment=False, **dataset_kwargs)
    test_dataset = EchoNetTemporalVariableStrideDataset(test_samples, augment=False, **dataset_kwargs)

    loader_kwargs = {
        'batch_size': base_config['batch_size'],
        'num_workers': base_config['num_workers'],
        'pin_memory': torch.cuda.is_available(),
        'persistent_workers': base_config['num_workers'] > 0,
    }
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader


def stride_config(temporal_stride: int) -> dict:
    config = dict(base_config)
    offsets = [offset * temporal_stride for offset in range(-(config['sequence_length'] // 2), config['sequence_length'] // 2 + 1)]
    config.update(
        {
            'temporal_stride': temporal_stride,
            'frame_offsets': offsets,
            'model_name': 'convlstm_unet',
            'selection_metric': 'val_dice',
            'checkpoint_selection': 'best validation Dice',
            'official_split': True,
            'test_policy': 'evaluate once after validation-selected checkpoint',
        }
    )
    return config


validation_rows = []
test_loaders = {}

for temporal_stride in TEMPORAL_STRIDES:
    print(f'\n=== Training temporal stride {temporal_stride} ===')
    run_dir = BASE_RUN_DIR / f'convlstm_unet_stride_{temporal_stride}'
    figures_dir = run_dir / 'figures'
    run_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    config = stride_config(temporal_stride)
    save_json(config, run_dir / 'config.json')
    train_loader, val_loader, test_loader = make_loaders(temporal_stride)
    test_loaders[temporal_stride] = test_loader

    sample_batch = next(iter(train_loader))
    print(f"Stride {temporal_stride} sequence shape: {tuple(sample_batch['sequence'].shape)}")
    print(f"Example frame indices: {sample_batch['frame_indices'][0].tolist()}")
    print(f"Example window span seconds: {float(sample_batch['window_span_seconds'][0]):.4f}")

    set_seed(config['seed'])
    model = build_convlstm_unet(
        in_channels=1,
        out_channels=1,
        channels=tuple(config['channels']),
    ).to(device)
    loss_fn = get_temporal_loss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
    )

    history = fit_temporal_v2(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=device,
        epochs=config['epochs'],
        output_dir=run_dir,
        early_stopping_patience=config['early_stopping_patience'],
        config=config,
    )
    plot_temporal_history(history, figures_dir / 'training_curves.png')

    best_row = history.sort_values('val_dice', ascending=False).iloc[0].to_dict()
    best_row.update(
        {
            'temporal_stride': temporal_stride,
            'run_dir': str(run_dir),
            'checkpoint_path': str(run_dir / 'checkpoints' / 'best_model.pt'),
        }
    )
    validation_rows.append(best_row)
    write_experiment_log(run_dir / 'experiment_log.md', config=config, best_val_metrics=best_row)

validation_df = pd.DataFrame(validation_rows).sort_values('val_dice', ascending=False)
validation_df.to_csv(BASE_RUN_DIR / 'stride_validation_comparison.csv', index=False)

baseline_validation_rows = []
baseline_history_path = ORIGINAL_BASELINE_RUN_DIR / 'history.csv'
if baseline_history_path.exists():
    baseline_history = pd.read_csv(baseline_history_path)
    if {'val_dice', 'val_iou', 'val_loss'}.issubset(baseline_history.columns):
        baseline_best = baseline_history.sort_values('val_dice', ascending=False).iloc[0].to_dict()
        baseline_best.update(
            {
                'temporal_stride': 1,
                'run_dir': str(ORIGINAL_BASELINE_RUN_DIR),
                'checkpoint_path': str(ORIGINAL_BASELINE_RUN_DIR / 'checkpoints' / 'best_model.pt'),
                'source': 'original_stride_1_baseline',
            }
        )
        baseline_validation_rows.append(baseline_best)

validation_with_baseline_df = pd.concat(
    [pd.DataFrame(baseline_validation_rows), validation_df.assign(source='variable_stride_experiment')],
    ignore_index=True,
).sort_values('temporal_stride')
validation_with_baseline_df.to_csv(BASE_RUN_DIR / 'stride_validation_comparison_with_baseline.csv', index=False)
validation_df[['temporal_stride', 'epoch', 'val_loss', 'val_dice', 'val_iou', 'val_precision', 'val_recall', 'checkpoint_path']]

## Validation-only stride ranking

In [ ]:
ranking_df = validation_df.reset_index(drop=True).copy()
ranking_df.insert(0, 'rank', range(1, len(ranking_df) + 1))
ranking_df.to_csv(BASE_RUN_DIR / 'stride_ranking.csv', index=False)

selected = ranking_df.iloc[0]
best_stride_summary = {
    'selected_stride': int(selected['temporal_stride']),
    'best_validation_dice': float(selected['val_dice']),
    'best_validation_iou': float(selected['val_iou']),
    'selected_checkpoint_path': selected['checkpoint_path'],
    'selection_criterion_used': 'highest validation Dice; held-out test metrics not used for selection',
}
save_json(best_stride_summary, BASE_RUN_DIR / 'best_stride_summary.json')

plot_validation_df = validation_with_baseline_df if len(validation_with_baseline_df) else validation_df
fig, axis = plt.subplots(figsize=(7, 4))
axis.plot(plot_validation_df['temporal_stride'], plot_validation_df['val_dice'], marker='o', label='Validation Dice')
axis.plot(plot_validation_df['temporal_stride'], plot_validation_df['val_iou'], marker='s', label='Validation IoU')
axis.set_xlabel('Temporal stride (frames)')
axis.set_ylabel('Score')
axis.set_ylim(0, 1)
axis.legend()
fig.tight_layout()
fig.savefig(BASE_RUN_DIR / 'validation_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)

print(best_stride_summary)
ranking_df[['rank', 'temporal_stride', 'val_dice', 'val_iou', 'val_loss', 'checkpoint_path']]

## Final held-out test evaluation

In [ ]:
test_rows = []

for row in validation_rows:
    temporal_stride = int(row['temporal_stride'])
    print(f'\n=== Final held-out test evaluation for stride {temporal_stride} ===')
    run_dir = BASE_RUN_DIR / f'convlstm_unet_stride_{temporal_stride}'
    config = stride_config(temporal_stride)
    _, _, test_loader = make_loaders(temporal_stride)

    model = build_convlstm_unet(
        in_channels=1,
        out_channels=1,
        channels=tuple(config['channels']),
    ).to(device)
    checkpoint_path = run_dir / 'checkpoints' / 'best_model.pt'
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

    loss_fn = get_temporal_loss()
    test_metrics = evaluate_temporal_test_v2(
        model=model,
        loader=test_loader,
        loss_fn=loss_fn,
        device=device,
        output_dir=run_dir,
        max_prediction_examples=config['max_prediction_examples'],
        max_area_curve_videos=config['max_area_curve_videos'],
    )
    test_metrics.update(
        {
            'temporal_stride': temporal_stride,
            'checkpoint_path': str(checkpoint_path),
            'selected_by': 'validation Dice',
        }
    )
    test_rows.append(test_metrics)
    write_experiment_log(run_dir / 'experiment_log.md', config=config, best_val_metrics=row, test_metrics=test_metrics)

test_df = pd.DataFrame(test_rows).sort_values('temporal_stride')
test_df.to_csv(BASE_RUN_DIR / 'stride_test_comparison.csv', index=False)

baseline_test_rows = []
baseline_test_path = ORIGINAL_BASELINE_RUN_DIR / 'test_metrics.json'
if baseline_test_path.exists():
    with baseline_test_path.open('r', encoding='utf-8') as file:
        baseline_test = json.load(file)
    baseline_test_rows.append(
        {
            'temporal_stride': 1,
            'test_loss': baseline_test.get('loss', baseline_test.get('test_loss')),
            'test_dice': baseline_test.get('dice', baseline_test.get('test_dice')),
            'test_iou': baseline_test.get('iou', baseline_test.get('test_iou')),
            'checkpoint_path': str(ORIGINAL_BASELINE_RUN_DIR / 'checkpoints' / 'best_model.pt'),
            'selected_by': 'original stride-1 baseline validation Dice',
            'source': 'original_stride_1_baseline',
        }
    )
test_with_baseline_df = pd.concat(
    [pd.DataFrame(baseline_test_rows), test_df.assign(source='variable_stride_experiment')],
    ignore_index=True,
).sort_values('temporal_stride')
test_with_baseline_df.to_csv(BASE_RUN_DIR / 'stride_test_comparison_with_baseline.csv', index=False)
test_df

## Final comparison plots and output checks

In [ ]:
plot_test_df = test_with_baseline_df if len(test_with_baseline_df) else test_df
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(plot_test_df['temporal_stride'], plot_test_df['test_dice'], marker='o', label='Test Dice')
axes[0].plot(plot_test_df['temporal_stride'], plot_test_df['test_iou'], marker='s', label='Test IoU')
axes[0].set_xlabel('Temporal stride (frames)')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].legend()

smoothness_col = 'temporal_mean_abs_area_change_px'
if smoothness_col in test_df:
    axes[1].plot(test_df['temporal_stride'], test_df[smoothness_col], marker='o')
    axes[1].set_ylabel('Mean abs LV area change (px)')
axes[1].set_xlabel('Temporal stride (frames)')
fig.tight_layout()
fig.savefig(BASE_RUN_DIR / 'test_and_temporal_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)

required_base_outputs = [
    BASE_RUN_DIR / 'stride_validation_comparison.csv',
    BASE_RUN_DIR / 'stride_test_comparison.csv',
    BASE_RUN_DIR / 'stride_ranking.csv',
    BASE_RUN_DIR / 'stride_validation_comparison_with_baseline.csv',
    BASE_RUN_DIR / 'stride_test_comparison_with_baseline.csv',
    BASE_RUN_DIR / 'best_stride_summary.json',
    BASE_RUN_DIR / 'validation_comparison.png',
    BASE_RUN_DIR / 'test_and_temporal_comparison.png',
]
missing = [path for path in required_base_outputs if not path.exists()]
for temporal_stride in TEMPORAL_STRIDES:
    run_dir = BASE_RUN_DIR / f'convlstm_unet_stride_{temporal_stride}'
    missing.extend(
        path for path in [
            run_dir / 'config.json',
            run_dir / 'history.csv',
            run_dir / 'experiment_log.md',
            run_dir / 'test_metrics.json',
            run_dir / 'checkpoints' / 'best_model.pt',
            run_dir / 'checkpoints' / 'final_model.pt',
            run_dir / 'figures' / 'training_curves.png',
            run_dir / 'test_sample_metrics.csv',
            run_dir / 'temporal_consistency_per_video.csv',
            run_dir / 'temporal_consistency_metrics_table.csv',
            run_dir / 'lv_area_curve_samples.csv',
        ]
        if not path.exists()
    )
assert not missing, f'Missing required outputs: {missing}'
print(f'All variable-stride outputs available at: {BASE_RUN_DIR.resolve()}')